In [ ]:
from numpy import rint
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from logger import configure_logging
import logging
from data_loaders.web_search import manual_web_search
from dotenv import load_dotenv
import os
from langchain.tools import tool
from data_loaders.pdf_ingestion import ingest_from_user_uploaded_pdfs, download_pdf_from_url


load_dotenv(override=True)
tavily_api_key = os.getenv("TAVILY_API_KEY")
nvidia_api_key = os.getenv("NVIDIA_API_KEY")

configure_logging()


### class SourceSchema(BaseModel):
- source_id: "Unique identifier for the source",
- url: Valid URL of the source"
- title: "Title of the source"
- source_type:"Type of the source"
- retrieval_timestamp: "ISO-8601 timestamp of retrieval"
- summary: "Short abstract or snippet"
- confidence_score: "Relevance/confidence score (0-10)"


# Start from pdf ingestion

In [ ]:
from logger import get_logger, log_tool_call, log_agent_action, log_warning

logger = get_logger(__name__)

In [ ]:
from data_loaders.semantic_scholar_loader import paper_search, snippet_search, get_paper_id, get_paper_citations, get_paper_references, get_paper

In [ ]:
query = "Deep Fake Detection"
paper_search_results = paper_search(query)


In [ ]:
snippet_search_result = snippet_search(query, limit=5)
print(snippet_search_result)

In [ ]:
title = snippet_search_result[0]['title']

In [ ]:
paper_id = get_paper_id(title)
print(f"Paper ID for '{title}': {paper_id}")

In [ ]:
get_paper(paper_id, fields="title,abstract,year,authors") 

In [ ]:
get_paper_references(paper_id)

In [ ]:
get_paper_citations(paper_id)

In [ ]:
len(paper_search_results)


In [ ]:
from data_loaders.pdf_ingestion import ingest_from_user_uploaded_pdfs

ingest_from_user_uploaded_pdfs("Research Paper")

In [ ]:

def download_pdf_from_url(url: str, save_dir: str = "pdf-from-url") -> str:
    """Download a PDF from a URL and persist it to local storage.

    This helper fetches a remote resource with streaming, validates that the
    response appears to be a PDF, and writes the binary content to disk in the
    target directory. It returns the saved file path for later ingestion or
    returns None when the download fails.

    Attributes:
        url (str): HTTP or HTTPS URL pointing to a PDF resource.
        save_dir (str): Local directory where the downloaded file is stored.

    Example:
        saved_path = download_pdf_from_url(
            "https://example.org/paper.pdf",
            save_dir="pdf-from-url"
        )
    """
    os.makedirs(save_dir, exist_ok=True)

    # Extract filename from URL or fallback
    parsed = urlparse(url)
    filename = os.path.basename(parsed.path) or "downloaded.pdf"

    if not filename.endswith(".pdf"):
        filename += ".pdf"

    file_path = os.path.join(save_dir, filename)

    try:
        response = requests.get(url, stream=True, timeout=15)
        response.raise_for_status()

        # Validate content type (important)
        content_type = response.headers.get("Content-Type", "")
        if "pdf" not in content_type.lower():
            raise ValueError(
                f"URL does not appear to be a PDF (Content-Type: {content_type})"
            )

        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        logger.info("Successfully downloaded PDF from %s to %s", url, file_path)
        return file_path

    except Exception as e:
        logger.info("Failed to download PDF from %s: %s", url, e)
        return None


In [1]:
from data_loaders.pdf_ingestion import ingest_pdf_from_user
ingest_pdf_from_user()


/home/tankaizokuo/Code/Research_Assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
03:52:00 [INFO] Found 5 PDF(s) in pdf-from-user
03:52:00 [INFO] Loading embedding model: BAAI/bge-base-en
03:52:00 [INFO] No device provided, using cpu
03:52:00 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
03:52:00 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en/b737bf5dcc6ee8bdc530531266b4804a5d77b5d8/modules.json "HTTP/1.1 200 OK"
03:52:01 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
03:52:01 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en/b73

In [2]:
from retriever import retrieve
retrieve()

03:52:39 [INFO] Loading embedding model: BAAI/bge-base-en
03:52:39 [INFO] No device provided, using cpu



── Test query: 'YOLO' ──


03:52:39 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
03:52:39 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en/b737bf5dcc6ee8bdc530531266b4804a5d77b5d8/modules.json "HTTP/1.1 200 OK"
03:52:39 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
03:52:39 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en/b737bf5dcc6ee8bdc530531266b4804a5d77b5d8/config_sentence_transformers.json "HTTP/1.1 200 OK"
03:52:39 [INFO] Loading SentenceTransformer model from BAAI/bge-base-en.
03:52:39 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
03:52:39 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en/b737bf5dcc6ee8b


[1] score=0.7877 | HemaX - A CNN-Based Approach for Anemia Detection from Conjunctiva Pallor Images (2026)
HemaX - A CNN-Based Approach for Anemia Detection from Conjunctiva Pallor Images Sohom Ghosal Soham Banerjee Sagnik Mazumdar Dept of ECE Dept of ECE Dept of CSE IIITNR, Raipur, India IIITNR, Raipur, India IIITNR, Raipur, India Dept of ECE Dept of ECE Dept of CSE Email: sohom22101@iiitnr.edu.in Email

[2] score=0.7843 | Counterfactual Reconstruction for Agronomic ()
2 Counterfactualexplanationworkflow Thisdualformulationpenalizesbothpixel-wisevenationdisplacement(MSE)and degradation of spatial patterns (SSIM), ensuring veins remain topologically invariant despite disease removal. Texture Preservation Loss: Using pre-trained VGG-19, we compute Gram matrices Gl =

[3] score=0.7822 | Counterfactual Reconstruction for Agronomic ()
Othermodelsthathavemadesignificantadvancementsintheareaofobjectdetection include the DEMM-YOLO model, which includes deformable attention mechanisms that ad